# Notebook 1 — Where does each PEFT method operate?

This notebook is a **minimal, visual PEFT anatomy guide with zoom-ins** for a handful of PEFT families.

We keep a tiny frozen transformer-like block fixed and ask:

- **Prompt / prefix methods:** how do trainable tokens modify the **input space**?
- **Adapters:** how do residual bottlenecks modify the **hidden-state / activation path**?
- **LoRA:** how do low-rank matrices modify the **weight space**?
- **BitFit:** what happens when we touch **only biases**?
- **Linear probing:** what if the encoder is frozen and only the **readout head** changes?

This notebook is intentionally small and pedagogical. It is designed to help participants form the right mental model before running larger experiments.

In [ ]:
# Optional install cell for Colab / fresh environments
# !pip install -q torch matplotlib pandas

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1) A tiny frozen block

We will use a tiny transformer-style block with:

- input embeddings `x`
- a single self-attention-style projection path
- a feed-forward path
- layer norms and residual connections

We will **freeze the base block**, then create variants that add trainable state in different places.

In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, d_model=32, d_hidden=64):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_hidden),
            nn.GELU(),
            nn.Linear(d_hidden, d_model),
        )

    def forward(self, x):
        # x: [batch, seq, d_model]
        h = self.ln1(x)
        q = self.q(h)
        k = self.k(h)
        v = self.v(h)
        attn = (q @ k.transpose(-1, -2)) / math.sqrt(q.size(-1))
        attn = attn.softmax(dim=-1)
        x = x + self.o(attn @ v)

        h = self.ln2(x)
        x = x + self.ffn(h)
        return x


base_block = TinyBlock().to(device)
for p in base_block.parameters():
    p.requires_grad = False

sum(p.numel() for p in base_block.parameters())

## 2) A single shared input batch

We create one synthetic "mini-batch" of token embeddings to probe the frozen block.

In [ ]:
B, T, D = 8, 12, 32
x = torch.randn(B, T, D, device=device)

with torch.no_grad():
    y_base = base_block(x)

print("input shape:", x.shape)
print("output shape:", y_base.shape)

## 3) PEFT variants as small wrappers

Each wrapper below isolates **where** the trainable change enters the computation.

### A. Linear probing
The encoder stays frozen. Only a task head changes.

### B. Soft prompt tuning
We prepend trainable prompt embeddings to the input sequence.  
This changes the **input / token space** before the frozen model processes anything.

### C. Adapter
We insert a small trainable bottleneck in the residual stream.  
This changes the **hidden activations** without replacing the pretrained weights.

### D. LoRA
We add a low-rank update to a frozen weight matrix.  
This changes the **effective weights** seen by the forward pass.

### E. BitFit
We only expose bias terms as trainable.

In [ ]:
class FrozenEncoderWithHead(nn.Module):
    def __init__(self, encoder, d_model=32, n_classes=3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        h = self.encoder(x)
        pooled = h.mean(dim=1)
        return self.head(pooled)


class SoftPromptWrapper(nn.Module):
    def __init__(self, encoder, prompt_len=4, d_model=32):
        super().__init__()
        self.encoder = encoder
        self.prompt = nn.Parameter(torch.randn(1, prompt_len, d_model) * 0.02)

    def forward(self, x):
        B = x.size(0)
        prompt = self.prompt.expand(B, -1, -1)
        x_prompt = torch.cat([prompt, x], dim=1)
        h = self.encoder(x_prompt)
        return h[:, self.prompt.size(1):, :]  # return only original token positions


class Adapter(nn.Module):
    def __init__(self, d_model=32, bottleneck=8):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class AdapterWrapper(nn.Module):
    def __init__(self, encoder, d_model=32, bottleneck=8):
        super().__init__()
        self.encoder = encoder
        self.adapter = Adapter(d_model=d_model, bottleneck=bottleneck)

    def forward(self, x):
        h = self.encoder(x)
        return self.adapter(h)


class LoRALinear(nn.Module):
    def __init__(self, frozen_linear: nn.Linear, rank=4, alpha=8.0):
        super().__init__()
        self.frozen = frozen_linear
        self.rank = rank
        self.alpha = alpha
        self.A = nn.Parameter(torch.randn(frozen_linear.in_features, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, frozen_linear.out_features))

        for p in self.frozen.parameters():
            p.requires_grad = False

    def forward(self, x):
        base = self.frozen(x)
        delta = x @ self.A @ self.B
        return base + (self.alpha / self.rank) * delta


class TinyBlockWithLoRA(nn.Module):
    def __init__(self, base_block: TinyBlock, rank=4):
        super().__init__()
        self.base = base_block
        self.q_lora = LoRALinear(base_block.q, rank=rank)

    def forward(self, x):
        h = self.base.ln1(x)
        q = self.q_lora(h)  # LoRA only on q projection
        k = self.base.k(h)
        v = self.base.v(h)
        attn = (q @ k.transpose(-1, -2)) / math.sqrt(q.size(-1))
        attn = attn.softmax(dim=-1)
        x = x + self.base.o(attn @ v)

        h = self.base.ln2(x)
        x = x + self.base.ffn(h)
        return x


class BitFitTinyBlock(nn.Module):
    def __init__(self, d_model=32, d_hidden=64):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.q = nn.Linear(d_model, d_model, bias=True)
        self.k = nn.Linear(d_model, d_model, bias=True)
        self.v = nn.Linear(d_model, d_model, bias=True)
        self.o = nn.Linear(d_model, d_model, bias=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_hidden, bias=True),
            nn.GELU(),
            nn.Linear(d_hidden, d_model, bias=True),
        )

        # start from the frozen block's weights
        with torch.no_grad():
            self.q.weight.copy_(base_block.q.weight)
            self.k.weight.copy_(base_block.k.weight)
            self.v.weight.copy_(base_block.v.weight)
            self.o.weight.copy_(base_block.o.weight)

            self.ln1.weight.copy_(base_block.ln1.weight)
            self.ln1.bias.copy_(base_block.ln1.bias)
            self.ln2.weight.copy_(base_block.ln2.weight)
            self.ln2.bias.copy_(base_block.ln2.bias)

            self.ffn[0].weight.copy_(base_block.ffn[0].weight)
            self.ffn[0].bias.copy_(base_block.ffn[0].bias)
            self.ffn[2].weight.copy_(base_block.ffn[2].weight)
            self.ffn[2].bias.copy_(base_block.ffn[2].bias)

        for name, p in self.named_parameters():
            p.requires_grad = ("bias" in name)

    def forward(self, x):
        h = self.ln1(x)
        q = self.q(h)
        k = self.k(h)
        v = self.v(h)
        attn = (q @ k.transpose(-1, -2)) / math.sqrt(q.size(-1))
        attn = attn.softmax(dim=-1)
        x = x + self.o(attn @ v)

        h = self.ln2(x)
        x = x + self.ffn(h)
        return x

In [ ]:
variants = {
    "soft_prompt": SoftPromptWrapper(base_block).to(device),
    "adapter": AdapterWrapper(base_block).to(device),
    "lora_q": TinyBlockWithLoRA(base_block).to(device),
    "bitfit": BitFitTinyBlock().to(device),
    "linear_probe_head": FrozenEncoderWithHead(base_block).to(device),
}

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

pd.DataFrame(
    [{"method": name, "trainable_params": count_trainable_params(model)}
     for name, model in variants.items()]
).sort_values("trainable_params")

## 4) One-step representational comparison

We do not train yet.  
We only ask: **if I introduce a trainable mechanism here, which representation does it directly perturb?**

We compare each variant's output to the frozen baseline using:

- relative L2 shift
- mean cosine similarity

In [ ]:
def rep_stats(y_ref, y_new):
    flat_ref = y_ref.reshape(-1, y_ref.size(-1))
    flat_new = y_new.reshape(-1, y_new.size(-1))
    rel_l2 = (flat_new - flat_ref).norm(dim=-1).mean() / (flat_ref.norm(dim=-1).mean() + 1e-8)
    cos = F.cosine_similarity(flat_ref, flat_new, dim=-1).mean()
    return float(rel_l2.detach().cpu()), float(cos.detach().cpu())

rows = []
with torch.no_grad():
    y_ref = base_block(x)

    for name, model in variants.items():
        if name == "linear_probe_head":
            # head output lives in label space, so skip representation comparison
            continue
        y_new = model(x)
        rel_l2, cos = rep_stats(y_ref, y_new)
        rows.append({"method": name, "relative_L2_shift": rel_l2, "mean_cosine_to_base": cos})

df_shift = pd.DataFrame(rows).sort_values("relative_L2_shift")
df_shift

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(df_shift["method"], df_shift["relative_L2_shift"])
plt.ylabel("Relative L2 shift vs frozen output")
plt.title("Which PEFT change perturbs the representation most directly?")
plt.xticks(rotation=20)
plt.show()

## 5) Visual zoom: where does the PEFT state live?

The next cells are meant as **teaching diagrams**. They zoom from the whole model into one projection layer.

The key idea:

- **Prompt tuning** adds trainable vectors before the frozen model.
- **Adapters** add a small trainable residual branch inside the hidden-state path.
- **LoRA** keeps the pretrained weight frozen, but adds a trainable low-rank update: $W_{eff}=W_0 + BA$.
- **BitFit / LN tuning** update small existing parameter subsets.
- **Linear probing** leaves the encoder unchanged and trains only the readout head.


In [ ]:
from matplotlib.patches import Rectangle, FancyArrowPatch


def add_box(ax, xy, w, h, text, fc="white", lw=1.5, fontsize=10):
    box = Rectangle(xy, w, h, facecolor=fc, edgecolor="black", linewidth=lw)
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box


def add_arrow(ax, start, end, text=None, rad=0.0):
    arrow = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=13,
        linewidth=1.3,
        connectionstyle=f"arc3,rad={rad}",
        color="black",
    )
    ax.add_patch(arrow)
    if text:
        ax.text((start[0]+end[0])/2, (start[1]+end[1])/2 + 0.18, text, ha="center", fontsize=9)


def draw_peft_map():
    fig, ax = plt.subplots(figsize=(12, 5.5))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 7)
    ax.axis("off")

    # Main pipeline
    add_box(ax, (0.5, 3.0), 1.5, 0.8, "input\ntokens")
    add_box(ax, (2.7, 3.0), 1.7, 0.8, "embedding\nspace")
    add_box(ax, (5.0, 3.0), 2.2, 0.8, "frozen\nTransformer block")
    add_box(ax, (8.0, 3.0), 1.7, 0.8, "pooled\nfeatures")
    add_box(ax, (10.3, 3.0), 1.2, 0.8, "task\nhead")

    for s, e in [((2.0, 3.4), (2.7, 3.4)), ((4.4, 3.4), (5.0, 3.4)), ((7.2, 3.4), (8.0, 3.4)), ((9.7, 3.4), (10.3, 3.4))]:
        add_arrow(ax, s, e)

    # PEFT intervention callouts
    add_box(ax, (2.2, 5.3), 2.3, 0.8, "soft prompt\ntrainable tokens", fc="#f3f3f3")
    add_arrow(ax, (3.35, 5.3), (3.35, 3.85), "prepend / condition")

    add_box(ax, (4.9, 1.2), 2.4, 0.8, "adapter\nbottleneck branch", fc="#f3f3f3")
    add_arrow(ax, (6.1, 3.0), (6.1, 2.0), "hidden-state residual")

    add_box(ax, (5.1, 5.3), 2.0, 0.8, "LoRA\nΔW = BA", fc="#f3f3f3")
    add_arrow(ax, (6.1, 5.3), (6.1, 3.85), "weight update")

    add_box(ax, (7.6, 5.3), 1.8, 0.8, "BitFit / LN\nsmall params", fc="#f3f3f3")
    add_arrow(ax, (8.5, 5.3), (7.0, 3.85), "parameter subset", rad=-0.2)

    add_box(ax, (9.9, 1.2), 1.8, 0.8, "linear probe\nhead only", fc="#f3f3f3")
    add_arrow(ax, (10.8, 2.0), (10.9, 3.0), "readout only")

    ax.text(6, 6.65, "Same frozen checkpoint, different intervention sites", ha="center", fontsize=14, fontweight="bold")
    plt.show()


draw_peft_map()


### Zoom-in: LoRA inside one frozen linear projection

For a normal linear layer, the frozen checkpoint uses a matrix $W_0$.

LoRA does **not** train all entries of $W_0$. Instead it trains two smaller matrices:

- $A \in \mathbb{R}^{r 	imes d_{in}}$ projects down to rank $r$
- $B \in \mathbb{R}^{d_{out} 	imes r}$ projects back up
- the effective layer becomes $x(W_0 + BA)^T$

So the trainable update is constrained to be low-rank. The rank $r$ is the main capacity knob.

In [ ]:
def draw_lora_zoom(d_in=12, d_out=10, rank=3):
    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 6)
    ax.axis("off")

    ax.text(7, 5.65, "LoRA zoom-in: frozen weight + low-rank trainable update", ha="center", fontsize=14, fontweight="bold")

    # Input and output vectors
    add_box(ax, (0.5, 2.6), 1.1, 0.8, "x")
    add_box(ax, (12.4, 2.6), 1.1, 0.8, "y")
    add_arrow(ax, (1.6, 3.0), (2.3, 3.0))
    add_arrow(ax, (11.7, 3.0), (12.4, 3.0))

    # Frozen path
    add_box(ax, (2.3, 2.1), 2.2, 1.8, f"frozen W₀\n{d_out} × {d_in}", fc="#ffffff", lw=2)
    add_arrow(ax, (4.5, 3.0), (5.4, 3.0), "base path")

    # LoRA branch
    add_box(ax, (2.3, 0.4), 1.6, 0.9, f"A\n{rank} × {d_in}", fc="#f3f3f3")
    add_box(ax, (4.5, 0.4), 1.6, 0.9, f"B\n{d_out} × {rank}", fc="#f3f3f3")
    add_arrow(ax, (1.05, 2.6), (3.1, 1.3), "parallel low-rank path", rad=0.25)
    add_arrow(ax, (3.9, 0.85), (4.5, 0.85))
    add_arrow(ax, (6.1, 0.85), (7.0, 2.45), "ΔW = BA", rad=-0.2)

    # Sum and output
    add_box(ax, (7.0, 2.45), 1.1, 1.1, "+")
    add_arrow(ax, (8.1, 3.0), (9.0, 3.0))
    add_box(ax, (9.0, 2.2), 2.7, 1.6, "effective layer\nW_eff = W₀ + BA")

    # Parameter comparison
    full = d_in * d_out
    lora = rank * d_in + d_out * rank
    ax.text(7, 1.55, f"Full update would train {full} numbers. LoRA trains {lora} numbers at rank r={rank}.", ha="center", fontsize=11)
    ax.text(7, 1.15, "For real models, this gap becomes large because d_in and d_out are often thousands.", ha="center", fontsize=10)
    plt.show()


draw_lora_zoom()


### Tiny numerical example: what does a LoRA update look like?

The cell below creates a frozen weight and a rank-$r$ LoRA update. The heatmaps show that LoRA can affect every output dimension, but through a structured low-rank update rather than an arbitrary full matrix.

In [ ]:
torch.manual_seed(3)
d_in, d_out, rank = 16, 16, 3
W0 = torch.randn(d_out, d_in)
A = torch.randn(rank, d_in) * 0.05
B = torch.randn(d_out, rank) * 0.05
delta_W = B @ A
W_eff = W0 + delta_W

print("Frozen W0 shape:   ", tuple(W0.shape))
print("LoRA A shape:      ", tuple(A.shape))
print("LoRA B shape:      ", tuple(B.shape))
print("Delta W = B @ A:   ", tuple(delta_W.shape))
print("Rank(Delta W):     ", int(torch.linalg.matrix_rank(delta_W)))
print("Trainable params:  ", A.numel() + B.numel(), "vs full matrix:", W0.numel())

for title, mat in [("Frozen weight W₀", W0), ("LoRA update ΔW = BA", delta_W), ("Effective weight W₀ + ΔW", W_eff)]:
    plt.figure(figsize=(4.2, 3.5))
    plt.imshow(mat.detach().cpu())
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(title)
    plt.xlabel("input dimension")
    plt.ylabel("output dimension")
    plt.show()



### Zoom-in: prompt tuning changes the input the frozen model sees

Prompt tuning does not edit the checkpoint. Instead, it prepends trainable vectors to the sequence.

A useful mental model:

- the pretrained model is a fixed machine
- the prompt vectors are learned context
- the task-specific state lives *before* the model, in token / embedding space

For vision models, visual prompt tuning is similar in spirit: learn pixels, patches, or prompt tokens that condition the frozen backbone.


In [ ]:

def draw_prompt_zoom(seq_len=8, prompt_len=3, d_model=10):
    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.set_xlim(0, 13)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.text(6.5, 5.55, "Prompt tuning zoom-in: trainable tokens before a frozen model", ha="center", fontsize=14, fontweight="bold")

    # Original tokens
    for i in range(seq_len):
        add_box(ax, (0.6 + i*0.55, 3.6), 0.42, 0.55, f"x{i+1}", fc="#ffffff", fontsize=8)
    ax.text(2.6, 4.45, "original input tokens", ha="center", fontsize=10)

    # Prompt tokens
    for i in range(prompt_len):
        add_box(ax, (0.6 + i*0.55, 2.35), 0.42, 0.55, f"p{i+1}", fc="#f3f3f3", fontsize=8)
    ax.text(1.4, 1.95, "trainable prompt", ha="center", fontsize=10)

    # Concatenated sequence
    add_arrow(ax, (4.9, 3.85), (5.8, 3.2), "concat")
    for i in range(prompt_len + seq_len):
        label = f"p{i+1}" if i < prompt_len else f"x{i-prompt_len+1}"
        fc = "#f3f3f3" if i < prompt_len else "#ffffff"
        add_box(ax, (6.0 + i*0.45, 2.9), 0.35, 0.5, label, fc=fc, fontsize=7)

    add_box(ax, (9.7, 2.55), 2.2, 1.2, "frozen\ntransformer", fc="#ffffff", lw=2)
    add_arrow(ax, (9.15, 3.15), (9.7, 3.15))
    add_box(ax, (12.2, 2.75), 0.65, 0.8, "h")
    add_arrow(ax, (11.9, 3.15), (12.2, 3.15))

    ax.text(6.45, 1.1, "Only p₁...pₘ are trained; the checkpoint is unchanged.", ha="center", fontsize=11)
    plt.show()

draw_prompt_zoom()


In [ ]:

# Tiny visual prompt object: rows are prompt tokens, columns are embedding dimensions.
torch.manual_seed(4)
prompt = torch.randn(4, 16) * 0.02
plt.figure(figsize=(6, 2.6))
plt.imshow(prompt)
plt.colorbar(fraction=0.046, pad=0.04)
plt.title("Soft prompt parameters: prompt_len × d_model")
plt.xlabel("embedding dimension")
plt.ylabel("prompt token")
plt.show()

print("Trainable prompt params:", prompt.numel())
print("Interpretation: capacity grows with prompt length × embedding dimension.")



### Zoom-in: adapters add a residual bottleneck inside the model

Adapters do not replace the pretrained block. They insert a small trainable branch in the hidden-state stream.

Typical adapter shape:

1. down-project hidden state to a small bottleneck
2. apply a nonlinearity
3. up-project back to model dimension
4. add the result back as a residual correction

This makes adapters easy to swap per task because the base model can remain frozen.


In [ ]:

def draw_adapter_zoom(d_model=12, bottleneck=4):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_xlim(0, 13)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.text(6.5, 5.55, "Adapter zoom-in: a small residual branch in activation space", ha="center", fontsize=14, fontweight="bold")

    add_box(ax, (0.6, 2.7), 1.0, 0.8, "h")
    add_box(ax, (2.2, 2.45), 2.2, 1.3, "frozen\nlayer / block", fc="#ffffff", lw=2)
    add_arrow(ax, (1.6, 3.1), (2.2, 3.1))
    add_arrow(ax, (4.4, 3.1), (5.2, 3.1), "base hidden state")

    # adapter branch
    add_arrow(ax, (1.1, 2.7), (3.1, 1.45), "copy h", rad=0.2)
    add_box(ax, (3.1, 0.85), 1.5, 0.8, f"down\n{d_model}→{bottleneck}", fc="#f3f3f3")
    add_box(ax, (5.0, 0.85), 1.2, 0.8, "GELU", fc="#f3f3f3")
    add_box(ax, (6.6, 0.85), 1.5, 0.8, f"up\n{bottleneck}→{d_model}", fc="#f3f3f3")
    add_arrow(ax, (4.6, 1.25), (5.0, 1.25))
    add_arrow(ax, (6.2, 1.25), (6.6, 1.25))
    add_arrow(ax, (8.1, 1.25), (8.8, 2.8), "residual Δh")

    add_box(ax, (8.7, 2.65), 0.7, 0.9, "+")
    add_arrow(ax, (5.2, 3.1), (8.7, 3.1))
    add_box(ax, (10.3, 2.7), 1.2, 0.8, "h + Δh")
    add_arrow(ax, (9.4, 3.1), (10.3, 3.1))

    params = d_model*bottleneck + bottleneck + bottleneck*d_model + d_model
    ax.text(6.5, 0.25, f"Trainable adapter params ≈ 2 × d_model × bottleneck = small; here about {params} including biases.", ha="center", fontsize=10)
    plt.show()

draw_adapter_zoom()


In [ ]:

# Adapter as a matrix-shaped bottleneck: down and up projections.
torch.manual_seed(5)
d_model, bottleneck = 16, 4
W_down = torch.randn(bottleneck, d_model) * 0.1
W_up = torch.randn(d_model, bottleneck) * 0.1
for title, mat in [("Adapter down projection", W_down), ("Adapter up projection", W_up)]:
    plt.figure(figsize=(4.6, 3.0))
    plt.imshow(mat)
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(title)
    plt.xlabel("input dimension")
    plt.ylabel("output dimension")
    plt.show()

print("Adapter bottleneck params:", W_down.numel() + W_up.numel())
print("Full d_model × d_model matrix would have:", d_model * d_model)



### Zoom-in: BitFit and LayerNorm tuning change tiny existing parameter subsets

Bias-only methods do not introduce a new branch. They select a very small subset of existing parameters and train only those.

For BitFit, the trainable objects are bias vectors such as:

- attention projection biases
- MLP biases
- classifier biases

LayerNorm tuning is similar in spirit: only the normalization scale and shift are updated.


In [ ]:

def draw_bitfit_zoom(d_model=12):
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 5.5)
    ax.axis("off")
    ax.text(6, 5.15, "BitFit zoom-in: freeze weights, move only bias vectors", ha="center", fontsize=14, fontweight="bold")

    add_box(ax, (0.7, 2.4), 1.0, 0.8, "x")
    add_box(ax, (2.4, 1.6), 2.4, 2.3, "frozen weight W\nlarge matrix", fc="#ffffff", lw=2)
    add_arrow(ax, (1.7, 2.8), (2.4, 2.8))
    add_box(ax, (5.7, 3.0), 1.9, 0.55, "trainable bias b", fc="#f3f3f3")
    add_arrow(ax, (4.8, 2.8), (5.3, 2.8))
    add_box(ax, (5.3, 2.35), 0.6, 0.9, "+")
    add_arrow(ax, (6.65, 3.0), (5.9, 3.0), "add")
    add_arrow(ax, (5.9, 2.8), (8.1, 2.8))
    add_box(ax, (8.1, 2.4), 1.2, 0.8, "y")

    ax.text(6, 0.8, "The expressive power is limited, but the number of trainable parameters is tiny.", ha="center", fontsize=11)
    plt.show()

draw_bitfit_zoom()


In [ ]:

# Visualize a full weight matrix versus a trainable bias vector.
torch.manual_seed(6)
W = torch.randn(16, 16)
b = torch.randn(16) * 0.05
plt.figure(figsize=(4, 3.2))
plt.imshow(W)
plt.colorbar(fraction=0.046, pad=0.04)
plt.title("Frozen weight matrix W")
plt.xlabel("input dimension")
plt.ylabel("output dimension")
plt.show()

plt.figure(figsize=(6, 1.5))
plt.imshow(b.view(1, -1))
plt.colorbar(fraction=0.046, pad=0.04)
plt.title("Trainable BitFit bias vector b")
plt.yticks([])
plt.xlabel("output dimension")
plt.show()

print("Trainable BitFit params:", b.numel(), "vs full matrix:", W.numel())



### Zoom-in: linear probing trains only the readout

Linear probing is the cleanest baseline: the encoder is frozen, and only the final classifier / regression head learns.

This tests a very important hypothesis:

> Are the pretrained representations already linearly separable enough for my task?

If yes, linear probing can be surprisingly strong. If no, it plateaus quickly because it cannot reshape the internal representation.


In [ ]:

def draw_linear_probe_zoom():
    fig, ax = plt.subplots(figsize=(11.5, 4.5))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 5)
    ax.axis("off")
    ax.text(6, 4.65, "Linear probing: frozen feature extractor, trainable readout", ha="center", fontsize=14, fontweight="bold")

    add_box(ax, (0.7, 2.0), 1.2, 0.8, "input")
    add_box(ax, (2.6, 1.55), 3.0, 1.7, "frozen pretrained\nencoder", fc="#ffffff", lw=2)
    add_box(ax, (6.5, 2.0), 1.4, 0.8, "features h")
    add_box(ax, (8.8, 1.7), 1.9, 1.4, "trainable\nlinear head", fc="#f3f3f3")
    add_box(ax, (11.0, 2.0), 0.8, 0.8, "ŷ")
    add_arrow(ax, (1.9, 2.4), (2.6, 2.4))
    add_arrow(ax, (5.6, 2.4), (6.5, 2.4))
    add_arrow(ax, (7.9, 2.4), (8.8, 2.4))
    add_arrow(ax, (10.7, 2.4), (11.0, 2.4))

    ax.text(6, 0.65, "Good first question: is a cheap linear boundary enough?", ha="center", fontsize=11)
    plt.show()

draw_linear_probe_zoom()



### Zoom-in: partial and full fine-tuning move the checkpoint itself

Partial fine-tuning and full fine-tuning are not always called PEFT, but they are important reference points.

- **Partial fine-tuning:** update a selected subset of pretrained layers, often later blocks or the head.
- **Full fine-tuning:** update all pretrained weights.

They are expressive, but they also carry higher memory, compute, and storage cost.


In [ ]:

def draw_finetuning_zoom():
    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.text(6, 5.6, "Fine-tuning reference points: which pretrained blocks are allowed to move?", ha="center", fontsize=14, fontweight="bold")

    # partial
    ax.text(3, 4.8, "Partial fine-tuning", ha="center", fontsize=12, fontweight="bold")
    for i in range(5):
        fc = "#f3f3f3" if i >= 3 else "#ffffff"
        label = f"block {i+1}\n" + ("train" if i >= 3 else "frozen")
        add_box(ax, (0.7 + i*0.95, 3.45), 0.8, 0.75, label, fc=fc, fontsize=8)
    add_box(ax, (5.8, 3.45), 0.9, 0.75, "head\ntrain", fc="#f3f3f3", fontsize=8)

    # full
    ax.text(3, 2.45, "Full fine-tuning", ha="center", fontsize=12, fontweight="bold")
    for i in range(5):
        add_box(ax, (0.7 + i*0.95, 1.1), 0.8, 0.75, f"block {i+1}\ntrain", fc="#f3f3f3", fontsize=8)
    add_box(ax, (5.8, 1.1), 0.9, 0.75, "head\ntrain", fc="#f3f3f3", fontsize=8)

    # explanation
    add_box(ax, (7.3, 3.25), 3.8, 1.2, "More trainable parameters\nmore capacity\nmore memory/storage", fc="#ffffff")
    add_box(ax, (7.3, 1.0), 3.8, 1.2, "Strong reference baseline\nbut task-specific checkpoints\ncan be large", fc="#ffffff")
    plt.show()

draw_finetuning_zoom()



### One-page visual summary

This is the table to keep in mind as you move into the later notebooks.


In [ ]:

summary = pd.DataFrame([
    {"method": "linear probing", "trainable state": "head", "acts on": "label/readout space", "main knob": "none / head size"},
    {"method": "prompt tuning", "trainable state": "prompt tokens", "acts on": "input / conditioning space", "main knob": "prompt length"},
    {"method": "adapters", "trainable state": "bottleneck modules", "acts on": "hidden activation path", "main knob": "bottleneck width"},
    {"method": "LoRA", "trainable state": "A and B matrices", "acts on": "effective weight space", "main knob": "rank r"},
    {"method": "BitFit / LN tuning", "trainable state": "biases / norm params", "acts on": "small existing parameter subset", "main knob": "which params"},
    {"method": "partial fine-tuning", "trainable state": "selected pretrained layers", "acts on": "checkpoint weights", "main knob": "which layers"},
    {"method": "full fine-tuning", "trainable state": "all weights", "acts on": "entire checkpoint", "main knob": "learning rate / regularization"},
])
summary


## 6) A small synthetic training exercise

Below we build a tiny sequence classification task so participants can see how each intervention can succeed or fail on the *same* frozen encoder.

The synthetic labels depend on a pattern in the sequence. That lets us observe:

- when changing only the head is enough
- when reshaping the input with prompts helps
- when weight-domain changes like LoRA are more expressive

In [ ]:
def make_synthetic_dataset(n=512, seq_len=12, d_model=32, n_classes=3):
    x = torch.randn(n, seq_len, d_model)
    # Labels are based on frozen-encoder pooled features.
    with torch.no_grad():
        h = base_block(x.to(device)).cpu().mean(dim=1)
    W = torch.randn(d_model, n_classes)
    y = (h @ W).argmax(dim=-1)
    return x, y

X, y = make_synthetic_dataset()
X_train, y_train = X[:400].to(device), y[:400].to(device)
X_val, y_val = X[400:].to(device), y[400:].to(device)
print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)

In [ ]:
def make_classifier(method):
    if method == "linear_probe":
        model = FrozenEncoderWithHead(base_block).to(device)
        for p in model.encoder.parameters():
            p.requires_grad = False
        return model

    if method == "soft_prompt":
        encoder = SoftPromptWrapper(base_block).to(device)
        model = nn.Sequential(
            encoder,
            nn.AdaptiveAvgPool1d(1),  # will apply after transpose below via wrapper later
        )
        raise NotImplementedError("Use the explicit train loop below.")

    raise ValueError(method)

To keep the code readable, the next cell uses small method-specific wrappers.

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(self, encoder, d_model=32, n_classes=3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        h = self.encoder(x)
        pooled = h.mean(dim=1)
        return self.head(pooled)


def build_method(method):
    if method == "linear_probe":
        model = SequenceClassifier(base_block).to(device)
        for p in model.encoder.parameters():
            p.requires_grad = False

    elif method == "soft_prompt":
        model = SequenceClassifier(SoftPromptWrapper(base_block)).to(device)

    elif method == "adapter":
        model = SequenceClassifier(AdapterWrapper(base_block)).to(device)

    elif method == "lora":
        model = SequenceClassifier(TinyBlockWithLoRA(base_block)).to(device)

    elif method == "bitfit":
        model = SequenceClassifier(BitFitTinyBlock()).to(device)
        for p in model.head.parameters():
            p.requires_grad = True

    else:
        raise ValueError(method)

    return model


def fit(method, epochs=40, lr=3e-3):
    model = build_method(method)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    history = []

    for epoch in range(epochs):
        model.train()
        logits = model(X_train)
        loss = F.cross_entropy(logits, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            train_acc = (model(X_train).argmax(dim=-1) == y_train).float().mean().item()
            val_logits = model(X_val)
            val_loss = F.cross_entropy(val_logits, y_val).item()
            val_acc = (val_logits.argmax(dim=-1) == y_val).float().mean().item()

        history.append({
            "epoch": epoch + 1,
            "method": method,
            "train_loss": float(loss.item()),
            "val_loss": float(val_loss),
            "train_acc": train_acc,
            "val_acc": val_acc,
            "trainable_params": count_trainable_params(model),
        })

    return pd.DataFrame(history)


methods = ["linear_probe", "soft_prompt", "adapter", "lora", "bitfit"]
runs = [fit(m, epochs=30) for m in methods]
hist = pd.concat(runs, ignore_index=True)
hist.tail()

In [ ]:
plt.figure(figsize=(8, 4))
for method, g in hist.groupby("method"):
    plt.plot(g["epoch"], g["val_acc"], label=method)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Synthetic task: same base encoder, different adaptation locations")
plt.legend()
plt.show()

## 7) Practical takeaway

The point is **not** that one method universally wins.

The point is to internalize **what is being changed**:

| Method | Trainable object | Where it acts | Mental model |
|---|---|---|---|
| Linear probing | classifier head | output / label space | "reuse features as-is" |
| Soft prompt tuning | prompt embeddings | input space | "steer the frozen model by changing what it sees" |
| Adapter | bottleneck residual | hidden-state path | "add a small specialist module" |
| LoRA | low-rank weight delta | weight space | "change how a layer computes, cheaply" |
| BitFit | biases only | parameter subset | "nudge pretrained behavior" |

### Rule-of-thumb hypotheses to test later

- **Linear probing** is the first baseline when you think the pretrained features are already close to your task.
- **Soft / prompt tuning** is attractive when you want many task-specific states while keeping the base weights fixed.
- **Adapters** are useful when you want modularity and clear insertion points.
- **LoRA** is a strong default when you want more expressivity than prompts or BitFit, but still far fewer trainable weights than full finetuning.
- **BitFit** is a tiny, cheap baseline for small or mild shifts.

In your workshop, this notebook can serve as the "**where do they operate?**" intuition pass before participants move to a real checkpoint and dataset.